In [ ]:
import numpy as np
import imgaug.augmenters as iaa

import numpy as np
import cv2 #import OpenCV
import matplotlib.pyplot as plt
from skimage.filters.rank import entropy
from skimage.morphology import disk
import fnmatch
import os





def load_batch_Covid(batch_idx):
    # dummy function, implement this
    # Return a numpy array of shape (N, height, width, #channels)
    # or a list of (height, width, #channels) arrays (may have different image
    # sizes).
    # Images should be in RGB for colorspace augmentations.
    # (cv2.imread() returns BGR!)
    # Images should usually be in uint8 with values from 0-255.
    
    workingpath="/home/cyril/liora/projet"
    normal_files="/COVID-19_Radiography_Dataset/Normal"
    covid_files="/COVID-19_Radiography_Dataset/COVID"
    lungOpacity_files="/COVID-19_Radiography_Dataset/Lung_Opacity"
    viralPneumonia_files="/COVID-19_Radiography_Dataset/Viral Pneumonia"
    size_img=299
    
    nb_files_Covid=len(fnmatch.filter(os.listdir(workingpath+covid_files+"/images/"), "*.png"))
    all_img_Covid = np.zeros((nb_files_Covid,size_img, size_img, 3 ),  dtype=np.uint8)
    
    for i in range(0, nb_files_Covid):
        img_Covid = cv2.imread(workingpath+covid_files +"/images/COVID-"+str(batch_idx+1)+".png", cv2.IMREAD_GRAYSCALE)  
        img_Covid=cv2.equalizeHist(img_Covid)
        img_Covid_mask = cv2.imread(workingpath+covid_files +"/masks/COVID-"+str(batch_idx+1)+".png", cv2.IMREAD_GRAYSCALE) # pour lire

        img_mask_resized = cv2.resize(img_Covid_mask, (img_Covid.shape[0], img_Covid.shape[1]), 0, 0, cv2.INTER_NEAREST)
        img_Covid_and_mask = cv2.bitwise_and(img_Covid,img_mask_resized)   
    
        backtorgb = cv2.cvtColor(img_Covid_and_mask,cv2.COLOR_GRAY2RGB)
        all_img_Covid[i,:,:,:] = backtorgb

    
    return all_img_Covid+ (batch_idx % 255)

def train_on_images(images):
    # dummy function, implement this
    pass

# Pipeline:
# (1) Crop images from each side by 1-16px, do not resize the results
#     images back to the input size. Keep them at the cropped size.
# (2) Horizontally flip 50% of the images.
# (3) Blur images using a gaussian kernel with sigma between 0.0 and 3.0.
seq = iaa.Sequential([
    iaa.Crop(px=(1, 16), keep_size=False),
    iaa.Fliplr(0.5),
    iaa.GaussianBlur(sigma=(0, 3.0))
])

for batch_idx in range(100):
    images = load_batch(batch_idx)
    images_aug = seq(images=images)  # done by the library
    train_on_images(images_aug)